# Flood Detection using Geospatial Foundation Models  

### Objective
This notebook implements an end-to-end flood detection workflow for India dataset using the IBM Prithvi geospatial foundation model, fine-tuned for semantic segmentation on multi-sensor Earth observation data.

The approach combines complementary satellite observations to improve flood detection reliability:

- EOS-4 SAR backscatter for flood sensitivity under cloud cover and adverse weather  
- Resourcesat-2 / 2A optical multispectral imagery for enhanced land–water discrimination  
 

### References
- IBM Geospatial Flood Detection : 
  https://github.com/ibm-granite/geospatial/tree/main/uki-flooddetection  

- IBM–NASA Prithvi Foundation Models for Earth Observation:  
  https://huggingface.co/collections/ibm-nasa-geospatial/prithvi-for-earth-observation


# Setup
Install terratoch - https://github.com/terrastackai/terratorch

Works well with python 3.11

In [ ]:
!python --version

In [ ]:
!pip install git+https://github.com/IBM/terratorch.git
# !python -m pip install terratorch

In [ ]:
# pip install terratorch==1.2.6

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import sys 
import os
import sys
import numpy as np
import torch

import terratorch
from terratorch.datamodules import MultiTemporalCropClassificationDataModule
from terratorch.tasks import SemanticSegmentationTask
from terratorch.datasets.transforms import FlattenTemporalIntoChannels, UnflattenTemporalFromChannels

import albumentations
from albumentations.pytorch import ToTensorV2

import lightning.pytorch as pl
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint
import rasterio

## Flood Dataset

**Satellite Data:** ISRO EOS-4 SAR and Resourcesat-2/2A optical imagery.  

- **EOS-4 SAR:** HH and HV polarizations  
- **Resourcesat-2/2A:** Green, Red, NIR, and SWIR bands  

**Inland Water Bodies:** Global Surface Water Explorer  
https://global-surface-water.appspot.com/#data  

**Flood Inundation Maps (India):** West Bengal, May 29, 2024, from the Bhuvan platform - https://bhuvan-app1.nrsc.gov.in/nfvas/

**Label Creation:** Flood labels are generated using a combination of water bodies and the flood inundation maps.  

**Temporal Alignment:** Data is preferably collected for the same locations and dates. A temporal mismatch of ±1 day is allowed if necessary.  

**Spatial Resolution:** 18 m, with 512×512 patches containing the following channels: Label, HH, HV, Band1 (Green), Band2 (Red), Band3 (NIR), Band4 (SWIR).  

This section introduces the dataset and prepares it for further analysis.


In [ ]:
dataset_path = '/kaggle/input/datasets/manipes/ibm-flood-dataset-phase-2/data/'

In [ ]:
datamodule = terratorch.datamodules.GenericNonGeoSegmentationDataModule(
    batch_size=4,
    num_workers=2,
    num_classes=3,

    # Define dataset paths
    train_data_root=dataset_path+'image',
    train_label_data_root=dataset_path+'label',
    val_data_root=dataset_path+'image',
    val_label_data_root=dataset_path+'label',
    test_data_root=dataset_path+'image',
    test_label_data_root=dataset_path+'label',

    # Define splits
    train_split=dataset_path+'split/train.txt',
    val_split=dataset_path+'split/val.txt',
    test_split=dataset_path+'split/test.txt',

    img_grep='*image.tif',
    label_grep='*label.tif',

    train_transform=[
        albumentations.D4(), # Random flips and rotation
        albumentations.pytorch.transforms.ToTensorV2(),
    ],
    val_transform=None,  # Using ToTensor() by default
    test_transform=None,
    
    # mean and stds of HH, HV, Band1 (Green), Band2 (Red), Band3 (NIR), Band4 (SWIR)
    means = [841.1257, 371.6175, 1734.1789, 1588.3142, 1742.8452, 1218.5551],
    stds = [473.7090, 170.3611, 623.0474, 612.8465, 564.5835, 528.0894],
    no_data_replace=0,
    no_label_replace=-1,
    # We use all six bands of the data, so we don't need to define dataset_bands and output_bands.
    predict_data_root = "/kaggle/input/datasets/manipes/ibm-flood-dataset-phase-2/data/prediction/image/"
)

# Setup train and val datasets
datamodule.setup("fit")

In [ ]:
# checking datasets train split size
train_dataset = datamodule.train_dataset
print(len(train_dataset))

# checking datasets validation split size
val_dataset = datamodule.val_dataset
len(val_dataset)



In [ ]:
# plotting a few samples
val_dataset.plot(val_dataset[0])
val_dataset.plot(val_dataset[1])

## Fine-tuning the IBM Prithvi Model

In this section, we fine-tune the IBM Prithvi geospatial foundation model for flood detection over India. The model, pretrained on global Earth observation data, is adapted to multi-sensor inputs including SAR and optical imagery, and trained to segment flooded areas accurately.

The fine-tuning process leverages high-resolution patches and flood labels derived from Bhuvan inundation maps and water body datasets. During training, the model learns India-specific flood patterns, improving its ability to identify inundated regions under diverse land cover and weather conditions.

Monitoring and checkpointing ensure the best-performing model is saved based on validation metrics.


In [ ]:
#Hyperparameters
EPOCHS = 1 # Set number of epochs; 1 for testing
BATCH_SIZE = 16 
LR = 2.0e-5
WEIGHT_DECAY = 0.1
HEAD_DROPOUT=0.1
FREEZE_BACKBONE = False

BANDS =[1,2,3,4,5,6]
NUM_FRAMES = 1
CLASS_WEIGHTS =[0.1,0.1,0.74]
SEED = 0
OUT_DIR='/kaggle/working/'


In [ ]:
SEED = 0

pl.seed_everything(SEED)

# Logger
logger = TensorBoardLogger(
    save_dir=OUT_DIR,
    name="test",
)

# Callbacks
checkpoint_callback = ModelCheckpoint(
    monitor="val/mIoU",
    mode="max",
    dirpath=os.path.join(OUT_DIR, "test", "checkpoints"),
    filename="best-checkpoint-{epoch:02d}-{val_loss:.2f}",
    save_top_k=1,
)

# Trainer
trainer = pl.Trainer(
    accelerator="auto",
    strategy="auto",
    devices="auto",
    precision="bf16-mixed",
    num_nodes=1,
    logger=logger,
    max_epochs=EPOCHS,
    check_val_every_n_epoch=1,
    log_every_n_steps=10,
    enable_checkpointing=True,
    callbacks=[checkpoint_callback],
    # limit_predict_batches=1,  # predict only in the first batch for generating plots
)

# DataModule
data_module = datamodule


# Model

decoder_args = dict(
    decoder="UperNetDecoder",
    decoder_channels=256,
    decoder_scale_modules=True,
)

necks = [
    #dict(
    #        name="SelectIndices",
    #        # indices=[2, 5, 8, 11]    # indices for prithvi_vit_100
    #       #indices=[5, 11, 17, 23],   # indices for prithvi_eo_v2_300
    #        # indices=[7, 15, 23, 31]  # indices for prithvi_eo_v2_600
    #    ),
    dict(
            name="ReshapeTokensToImage",
            effective_time_dim=NUM_FRAMES,
        )
    ]

backbone_args = dict(
    backbone_pretrained=True,
    backbone="prithvi_eo_v2_300_tl", # other models are availble like prithvi_eo_v2_300, prithvi_eo_v2_tiny_tl, prithvi_eo_v2_600, prithvi_eo_v2_600_tl
    #backbone_coords_encoding=["time", "location"],
    backbone_bands=BANDS,
    backbone_num_frames=1, # 1 is the default value,    
    # backbone_pretrained_cfg_overlay=None
    )

model_args = dict(
    **backbone_args,
    **decoder_args,
    num_classes=3,
    head_dropout=HEAD_DROPOUT,
    necks=necks,
    rescale=True,
)
    

model = SemanticSegmentationTask(
    model_args=model_args,
    plot_on_val=False,
    class_weights=CLASS_WEIGHTS,
    loss="ce",
    lr=LR,
    optimizer="AdamW",
    optimizer_hparams=dict(weight_decay=WEIGHT_DECAY),
    ignore_index=-1,
    freeze_backbone=FREEZE_BACKBONE,
    freeze_decoder=False,
    model_factory="EncoderDecoderFactory",
)



In [ ]:
trainer.fit(model, datamodule=data_module)

## Testing

This section evaluates the fine-tuned Prithvi model on unseen flood events. 

In [ ]:
!ls /kaggle/working/test/checkpoints/

In [ ]:
datamodule.setup("test")
#sample checkpoint from 37th epoch for testing..
best_ckpt_path = "/kaggle/working/test/checkpoints/best-checkpoint-epoch=00-val_loss=0.00.ckpt"

In [ ]:
test_results = trainer.test(model, datamodule=datamodule, ckpt_path=best_ckpt_path)

## Prediction

This section runs inference using the fine-tuned model and generates flood predictions for unseen data. The predicted flood masks are saved as georeferenced GeoTIFF files, preserving the spatial metadata of the input imagery for downstream analysis and visualization.


In [ ]:
datamodule.setup("predict")

predictions = trainer.predict(
    model,
    datamodule=datamodule,
    ckpt_path=best_ckpt_path
)

output_dir = "/kaggle/working/prediction"
os.makedirs(output_dir, exist_ok=True)

for batch_idx, (preds, file_paths) in enumerate(predictions):
    
    if isinstance(preds, tuple):
        preds = preds[0]

    if preds.ndim == 4:               # [B, C, H, W]
        preds = preds.argmax(dim=1)   # [B, H, W]

    preds = preds.cpu().numpy().astype("int16")

    for i in range(preds.shape[0]):
        arr = preds[i]
        arr[arr < 0] = -1             # 🔒 enforce ignore_index

        ref_path = file_paths[i]
        with rasterio.open(ref_path) as src:
            meta = src.meta.copy()

        meta.update({
            "count": 1,
            "dtype": "int16",
            "nodata": -1,
            "compress": "lzw",
        })

        out_name = (
            os.path.splitext(os.path.basename(ref_path))[0]
            + ".tif"
        )
        out_path = os.path.join(output_dir, out_name)

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(arr, 1)

        print(f"Saved {out_path}")


# Convert prediction tif files to Kaggle-style run-length encoding (RLE) Submission csv

In [ ]:
import numpy as np

def mask_to_rle(mask):
    """
    Convert binary mask to RLE (Kaggle format).
    Mask must be 2D numpy array with values 0 or 1.
    """
    pixels = mask.flatten(order="F")  # column-major
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)


In [ ]:
import rasterio
import pandas as pd
from pathlib import Path

tif_dir = Path("/kaggle/working/prediction")   # folder with .tif files
output_csv = "/kaggle/working/prediction/submission.csv"

rows = []

for tif_path in sorted(tif_dir.glob("*.tif")):
    with rasterio.open(tif_path) as src:
        mask = src.read(1)

    # Convert to binary
    mask = (mask == 1).astype(np.uint8)

    rle = mask_to_rle(mask)

    rows.append({
        "id": tif_path.name.replace("_image.tif", ""), 
        "rle_mask": rle
    })

df = pd.DataFrame(rows)
df = df.replace("", 0).fillna(0) #replace null/ na with zero - kaggle compatible
df.to_csv(output_csv, index=False)
print(f"Saved Kaggle RLE CSV : {output_csv}")
